In [1]:
import numpy as np
import pandas as pd
from faker import Faker

In [2]:
fake = Faker()

In [3]:
user_data = []

for _ in range(10_000):
    join_date = fake.date_between(start_date='-15y', end_date='now')
    user_variant = fake.random_element(elements=('A', 'B'))
    user_data.append({ 
        'user_id': fake.uuid4(),
        'join_date': join_date,
        'variant_group': user_variant
    })

users = pd.DataFrame(user_data)

users.head(10)

,user_id,join_date,variant_group
0,7f79d868-178d-41d0-950d-d3dba2a1d275,2021-08-15,A
1,30370b9d-d1b3-4494-beca-c06993b02326,2020-12-16,A
2,2ddad9c7-3ef7-4880-8b3a-e9fda1cc21bc,2012-02-27,A
3,6c16803c-778c-4360-94b7-343fe25551b7,2022-08-02,A
4,57db7805-d308-470a-9297-ca3b37f0f0f7,2018-11-05,A
5,c0ece308-f803-45da-8d0d-bffc1a75e9e8,2022-12-17,A
6,5ca3586a-02f0-48f4-9fac-ea0b14f38012,2021-08-07,A
7,013db783-c29a-4ddc-8dff-343e3283b0f5,2016-04-30,A
8,d08227fa-299e-4944-83f5-461f2c672c99,2021-12-30,B
9,f2f13ab8-e14a-4112-b5d3-0da809e168f9,2014-02-22,A


In [4]:
num_events = 25_000

events = pd.DataFrame({
    'event_id': [fake.uuid4() for _ in range(num_events)],
    'user_id': np.random.choice(users['user_id'], size=num_events),
    'session_duration': np.random.randint(10, 600, size=num_events)
})

events = events.merge(users[['user_id', 'variant_group', 'join_date']], on='user_id', how='left')

events['days_since_join'] = np.random.randint(0, 30, size=num_events)
events['event_date'] = pd.to_datetime(events['join_date']) + pd.to_timedelta(events['days_since_join'], unit='d')

rev_choices = [0, 9.99, 19.99]
rev_probs_A = [0.85, 0.12, 0.03]
rev_probs_B = [0.78, 0.15, 0.07] 

rev_A = np.random.choice(rev_choices, size=num_events, p=rev_probs_A)
rev_B = np.random.choice(rev_choices, size=num_events, p=rev_probs_B)

events['revenue_generated'] = np.where(events['variant_group'] == 'A', rev_A, rev_B)

events = events.drop(columns=['variant_group', 'join_date', 'days_since_join'])

events.head()

,event_id,user_id,session_duration,event_date,revenue_generated
0,1e92c710-c085-4347-8893-7f113e07b6cc,29c9ae95-280c-43b9-89b1-be720f384616,288,2014-06-04,0.0
1,4cfbe51f-18fc-481f-a662-ffcfeb7470c5,7735a121-c0b5-4c4a-8df6-b17738255806,393,2021-11-09,0.0
2,0332ef21-77a8-4a4a-a1c2-824f612c0190,3ff9374c-b343-4558-9711-518875ad078b,180,2025-05-09,0.0
3,d51898e2-7319-4a23-bdc1-cc2f40ca9a68,ec3fda2f-e736-446a-8aa8-cc4882fd6f0f,596,2014-09-26,0.0
4,3b2ae34c-1186-4109-aa5f-eb35cc9c49f2,d7126a78-7eac-4832-972f-129084edaa84,252,2013-11-07,0.0


In [5]:
users.to_csv('users.csv', index=False)
events.to_csv('events.csv', index=False)